# Project Aura 2.0 — Notebook 05: Gesture Recognition Engine

## Overview

Implements gesture recognition with MediaPipe and OpenCV. Detects hand landmarks, classifies gestures, smooths predictions, produces structured events for Notebook 06.

## Sections
1. Environment & Checks
2. Imports & Config
3. MediaPipe & Hand Tracking
4. Landmark Processing
5. Gesture Classification
6. Temporal Smoothing
7. Event Generation
8. Image Demo
9. Video Demo
10. Webcam Fallback
11. Visualization
12. Tests
13. Integration
14. Export & Cleanup
15. Summary

In [ ]:
!pip install --quiet mediapipe opencv-python-headless numpy pillow matplotlib
print("Dependencies installed for NB05")

## 1. Environment & Dependency Checks

In [ ]:
import os, sys
print(f"Python: {sys.version}")
IN_COLAB = "google.colab" in sys.modules
print(f"Colab: {IN_COLAB}")
for name, disp in [("cv2","OpenCV"),("mediapipe","MediaPipe"),("numpy","NumPy"),("torch","PyTorch")]:
    try:
        m = __import__(name)
        print(f"  {disp}: {getattr(m,'__version__','?')}")
    except ImportError:
        print(f"  {disp}: NOT INSTALLED")
try:
    import torch
    print(f"  CUDA: {torch.cuda.is_available()}")
except ImportError:
    print("  CPU fallback (torch not installed)")

## 2. Imports & Configuration

In [ ]:
PROJECT_DIR = os.getcwd()
for p in [os.path.dirname(PROJECT_DIR) if "project-aura" in PROJECT_DIR else PROJECT_DIR, "/content/project-aura", PROJECT_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.core.config import load_config
from src.vision.hand_tracker import HandTracker
from src.vision.landmark_processor import validate_landmarks, compute_bounding_box, extended_fingers
from src.vision.gesture_recognizer import GestureRecognizer, GesturePrediction, OPEN_PALM, FIST, THUMBS_UP, POINTING, VICTORY, PINCH, UNKNOWN
from src.vision.gesture_smoother import GestureSmoother
from src.vision.gesture_events import GestureEventGenerator

config = load_config()
print(f"Config: {config.vision.model_name}, threshold={config.gesture.confidence_threshold}")

## 3. MediaPipe & Hand Tracking

In [ ]:
tracker = HandTracker(max_num_hands=config.gesture.max_num_hands)
print(f"MediaPipe available: {tracker.available}")
if not tracker.available:
    print("Install: pip install mediapipe")

## 4. Landmark Processing

In [ ]:
def _palm():
    lm = [(0.5,0.5,0.0)]*21
    lm[4]=(0.7,0.5,0.0);lm[3]=(0.4,0.5,0.0)
    lm[8]=(0.5,0.2,0.0);lm[7]=(0.5,0.5,0.0)
    lm[12]=(0.5,0.2,0.0);lm[11]=(0.5,0.5,0.0)
    lm[16]=(0.5,0.2,0.0);lm[15]=(0.5,0.5,0.0)
    lm[20]=(0.5,0.2,0.0);lm[19]=(0.5,0.5,0.0)
    return lm

lm = _palm()
print(f"Valid: {validate_landmarks(lm)}")
print(f"Extended: {extended_fingers(lm)}")
print(f"BBox: {compute_bounding_box(lm, 640, 480)}")

## 5. Gesture Classification

Rule-based geometric recognition. Not a neural network.

In [ ]:
rec = GestureRecognizer(confidence_threshold=config.gesture.confidence_threshold)

def _fist():
    lm=[(0.5,0.5,0.0)]*21
    lm[4]=(0.3,0.9,0.0);lm[3]=(0.4,0.5,0.0)
    lm[8]=(0.5,0.9,0.0);lm[7]=(0.5,0.5,0.0)
    lm[12]=(0.5,0.9,0.0);lm[11]=(0.5,0.5,0.0)
    lm[16]=(0.5,0.9,0.0);lm[15]=(0.5,0.5,0.0)
    lm[20]=(0.5,0.9,0.0);lm[19]=(0.5,0.5,0.0)
    return lm

def _thumb():
    lm=[(0.5,0.5,0.0)]*21
    lm[4]=(0.7,0.3,0.0);lm[3]=(0.4,0.5,0.0)
    lm[8]=(0.5,0.9,0.0);lm[7]=(0.5,0.5,0.0)
    lm[12]=(0.5,0.9,0.0);lm[11]=(0.5,0.5,0.0)
    lm[16]=(0.5,0.9,0.0);lm[15]=(0.5,0.5,0.0)
    lm[20]=(0.5,0.9,0.0);lm[19]=(0.5,0.5,0.0)
    return lm

def _point():
    lm=[(0.5,0.5,0.0)]*21
    lm[4]=(0.3,0.5,0.0);lm[3]=(0.4,0.5,0.0)
    lm[8]=(0.5,0.2,0.0);lm[7]=(0.5,0.5,0.0)
    lm[12]=(0.5,0.9,0.0);lm[11]=(0.5,0.5,0.0)
    lm[16]=(0.5,0.9,0.0);lm[15]=(0.5,0.5,0.0)
    lm[20]=(0.5,0.9,0.0);lm[19]=(0.5,0.5,0.0)
    return lm

def _victory():
    lm=[(0.5,0.5,0.0)]*21
    lm[4]=(0.3,0.5,0.0);lm[3]=(0.4,0.5,0.0)
    lm[8]=(0.5,0.2,0.0);lm[7]=(0.5,0.5,0.0)
    lm[12]=(0.5,0.2,0.0);lm[11]=(0.5,0.5,0.0)
    lm[16]=(0.5,0.9,0.0);lm[15]=(0.5,0.5,0.0)
    lm[20]=(0.5,0.9,0.0);lm[19]=(0.5,0.5,0.0)
    return lm

def _pinch():
    lm=[(0.5,0.5,0.0)]*21
    lm[4]=(0.5,0.3,0.0);lm[8]=(0.5,0.3,0.0)
    lm[12]=(0.5,0.9,0.0);lm[16]=(0.5,0.9,0.0);lm[20]=(0.5,0.9,0.0)
    return lm

tests=[("OPEN_PALM",_palm()),("FIST",_fist()),("THUMBS_UP",_thumb()),("POINTING",_point()),("VICTORY",_victory()),("PINCH",_pinch())]
for name,lm in tests:
    p=rec.recognize(lm,handedness="Right")
    print(f"  [{'OK' if p.gesture==name else 'FAIL'}] {name}: {p.gesture} ({p.confidence:.2f})")

In [ ]:
p=rec.recognize_many([([(0.5,0.5)]*5,"Right")])
print(f"Invalid: {p[0].gesture}")
p2=rec.recognize(_palm(),handedness="Left")
print(f"Left palm: {p2.gesture}")

## 6. Temporal Smoothing

In [ ]:
sm = GestureSmoother(window_size=5, min_stable_frames=3, cooldown_frames=10)
for i,g in enumerate([OPEN_PALM]*5+[FIST]*5):
    r=sm.update(GesturePrediction(gesture=g,confidence=0.9,handedness="Right"))
    if r.transition:
        print(f"  Frame {i}: TRANSITION -> {r.gesture}")
print(f"Final: {r.gesture}")

## 7. Event Generation

In [ ]:
import json
gen = GestureEventGenerator(source="demo")
events=[]
for g in [OPEN_PALM,OPEN_PALM,FIST,FIST,VICTORY]:
    events.append(gen.create_event(gesture=g,confidence=0.9,hand="Right",bbox=[100,100,300,400]))
print(json.dumps(events[0].to_dict(),indent=2,default=str)[:600])
for e in events:
    d=e.to_dict();assert "event_id" in d and "gesture" in d and "confidence" in d
print("Schema OK")

## 8. Image Demo

In [ ]:
try:
    import numpy as np
    from PIL import Image,ImageDraw
    img=Image.new("RGB",(640,480),(50,50,50))
    d=ImageDraw.Draw(img)
    d.ellipse([270,180,370,280],fill=(200,180,160))
    for i in range(5):
        d.rectangle([280+i*20-5,100,280+i*20+5,170],fill=(200,180,160))
    arr=np.array(img)
    res=tracker.detect(arr)
    print(f"Hands: {res.num_hands}, available: {res.tracker_available}")
except Exception as e:
    print(f"Image demo skipped: {e}")

## 9. Video Demo

In [ ]:
vs=GestureSmoother(window_size=5,min_stable_frames=2,cooldown_frames=3)
vg=GestureEventGenerator(source="video")
for i,g in enumerate([OPEN_PALM,OPEN_PALM,OPEN_PALM,FIST,FIST,FIST,FIST]):
    r=vs.update(GesturePrediction(gesture=g,confidence=0.9))
    if r.transition:
        ev=vg.create_event(gesture=r.gesture,confidence=r.confidence)
        print(f"  Frame {i}: TRANSITION -> {ev.gesture}")
    else:
        print(f"  Frame {i}: {r.gesture}")

## 10. Webcam Fallback

Colab does not support `cv2.VideoCapture(0)` for user webcams. Use image/video upload to `/content/`.

In [ ]:
print("Use uploaded images/videos for Colab. Local webcam works with cv2.VideoCapture(0).")

## 11. Visualization

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(15,5))
fig.suptitle("Synthetic Landmarks")
for ax,(n,lm) in zip(axes,tests[:3]):
    xs=[p[0] for p in lm];ys=[p[1] for p in lm]
    ax.scatter(xs,ys,c="blue",s=30);ax.plot(xs,ys,"b-",alpha=0.3)
    ax.set_title(n);ax.set_xlim(0,1);ax.set_ylim(1,0)
plt.tight_layout();plt.show()

## 12. Unit Tests

In [ ]:
print("Running tests...")
assert validate_landmarks(_palm())
assert rec.recognize(_fist()).gesture==FIST
assert rec.recognize(_palm()).gesture==OPEN_PALM
assert rec.recognize_many([([(0.5,0.5)]*5,"Right")])[0].gesture==UNKNOWN
s=GestureSmoother(window_size=3,min_stable_frames=2,cooldown_frames=0)
for _ in range(3): s.update(GesturePrediction(gesture=OPEN_PALM,confidence=0.9))
assert s.update(GesturePrediction(gesture=OPEN_PALM,confidence=0.9)).gesture==OPEN_PALM
e=GestureEventGenerator(source="t").create_event(gesture=OPEN_PALM,confidence=0.9)
assert e.event_id and e.gesture==OPEN_PALM
print("All passed")

## 13. Integration

In [ ]:
pt=HandTracker();pr=GestureRecognizer();ps=GestureSmoother(window_size=3,min_stable_frames=2,cooldown_frames=0);pg=GestureEventGenerator(source="int")
for i in range(5):
    p=pr.recognize(_palm(),handedness="Right")
    s=ps.update(p)
    if s.transition:
        ev=pg.create_event(gesture=s.gesture,confidence=s.confidence)
        print(f"  Frame {i}: {ev.gesture}")
print(f"Tracker: {pt.available}")

## 14. Export & Cleanup

In [ ]:
os.makedirs("output",exist_ok=True)
with open("output/gesture_events.json","w") as f:
    json.dump([e.to_dict() for e in events],f,indent=2,default=str)
print(f"Exported {len(events)} events")
tracker.release();pt.release()
print("Cleaned up")

## 15. Summary

Hand tracking, geometric gesture recognition (7 classes), temporal smoothing, structured events. Event schema documented for Notebook 06.